In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from datetime import datetime, timedelta, date
import calendar
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def second_thursday(year, month):
    """Return the second Thursday of a given month/year."""
    first_day = date(year, month, 1)
    # Thursday = 3 in weekday()
    day_of_week = first_day.weekday()
    first_thu = 1 + (3 - day_of_week) % 7
    second_thu = first_thu + 7
    return date(year, month, second_thu)


def generate_meeting_dates(start_date, n=14):
    """Generate next n BCRP meeting dates (2nd Thursday each month) from start_date."""
    meetings = []
    y, m = start_date.year, start_date.month
    while len(meetings) < n:
        d = second_thursday(y, m)
        if d >= start_date:
            meetings.append(d)
        m += 1
        if m > 12:
            m = 1
            y += 1
    return meetings


def pen_ois_fair_value(tibo_now, meeting_dates, path_bps, settle_date, maturity_date):
    """
    Compute PEN OIS fair value (zero coupon, ACT/360, annual, nominal rate).
    
    The OIS references TIBO overnight. Under no-arbitrage the fixed rate on a
    zero-coupon OIS equals the annualised compounded overnight rate.
    
    For PEN OIS (quoted as nominal annual ACT/360):
      FV_rate = [ product(1 + r_i / 360) - 1 ] * 360 / total_days
    where r_i is the overnight rate on each business day.
    
    We simplify by compounding the reference rate that steps on meeting dates.
    """
    total_days = (maturity_date - settle_date).days
    if total_days <= 0:
        return np.nan
    
    # Build rate schedule: list of (start_date, rate) pairs
    # Rate changes effective the day AFTER the meeting (post 5pm fix)
    rate = tibo_now / 100.0  # convert to decimal
    schedule = [(settle_date, rate)]
    
    for i, mtg in enumerate(meeting_dates):
        effective = mtg + timedelta(days=1)  # change effective next day
        if effective <= settle_date:
            # This meeting already happened, apply its change to starting rate
            rate += path_bps[i] / 10000.0
            schedule = [(settle_date, rate)]
            continue
        if effective >= maturity_date:
            break
        rate += path_bps[i] / 10000.0
        schedule.append((effective, rate))
    
    # Compound overnight
    compounded = 1.0
    for j in range(len(schedule)):
        seg_start = schedule[j][0]
        seg_rate = schedule[j][1]
        if j + 1 < len(schedule):
            seg_end = schedule[j + 1][0]
        else:
            seg_end = maturity_date
        n_days = (seg_end - seg_start).days
        if n_days > 0:
            compounded *= (1 + seg_rate * n_days / 360.0)
    
    # Convert back to nominal annual rate ACT/360
    fv_rate = (compounded - 1) * 360.0 / total_days * 100.0
    return fv_rate


def generate_simulated_history(current_rate, n_days=365, vol_bps=3.0):
    """Generate simulated 1y lookback of OIS rate with mean reversion."""
    np.random.seed(42)
    rates = np.zeros(n_days)
    # Start higher (simulate easing cycle)
    rates[0] = current_rate + 0.80
    dt = 1.0 / 252
    kappa = 2.0  # mean reversion speed
    sigma = vol_bps / 100.0
    for i in range(1, n_days):
        dr = kappa * (current_rate - rates[i-1]) * dt + sigma * np.sqrt(dt) * np.random.randn()
        rates[i] = rates[i-1] + dr
    dates = pd.bdate_range(end=date.today(), periods=n_days)
    return dates, rates

print('Helpers loaded.')

In [ ]:
# ============================================================
# PEN OIS FAIR VALUE MODEL — INTERACTIVE
# ============================================================

# --- STYLING ---
DARK_BG = '#0a1628'
PANEL_BG = '#0d1f3c'
HEADER_BG = '#122a4f'
ACCENT = '#00e5ff'
GREEN = '#00c853'
RED = '#ff1744'
WHITE = '#e0e0e0'
GOLD = '#ffd600'

CSS = f"""
<style>
.pen-ois-container {{ font-family: 'Consolas', 'Courier New', monospace; color: {WHITE}; background: {DARK_BG}; padding: 10px; }}
.pen-ois-title {{ color: {ACCENT}; font-size: 20px; font-weight: bold; margin-bottom: 8px; }}
.pen-ois-subtitle {{ color: {GOLD}; font-size: 14px; font-weight: bold; margin: 6px 0; }}
table.fv-table {{ border-collapse: collapse; width: 100%; font-size: 12px; }}
table.fv-table th {{ background: {HEADER_BG}; color: {ACCENT}; padding: 5px 8px; border: 1px solid #1a3a5c; text-align: center; }}
table.fv-table td {{ background: {PANEL_BG}; color: {WHITE}; padding: 4px 8px; border: 1px solid #1a3a5c; text-align: center; }}
table.fv-table tr.fv-row td {{ background: #1a3a5c; font-weight: bold; }}
table.fv-table tr.delta-row td {{ font-weight: bold; }}
.pos {{ color: {GREEN}; }} .neg {{ color: {RED}; }}
</style>
"""

# --- GLOBAL STATE ---
TODAY = date.today()  # 2026-03-09
SETTLE = TODAY + timedelta(days=1)  # T+1 settlement
MEETING_DATES = generate_meeting_dates(TODAY, n=14)

# Tenor definitions: (label, months)
TENORS = [('3M', 3), ('6M', 6), ('9M', 9), ('12M', 12)]

def maturity_for_tenor(settle, months):
    y = settle.year + (settle.month + months - 1) // 12
    m = (settle.month + months - 1) % 12 + 1
    d = min(settle.day, calendar.monthrange(y, m)[1])
    return date(y, m, d)

MATURITIES = {label: maturity_for_tenor(SETTLE, mo) for label, mo in TENORS}

# --- DEFAULT PATHS (12 scenarios) ---
DEFAULT_PATHS = {
    'Hold':           [0]*14,
    '-25 Apr':        [0, -25] + [0]*12,
    '-25 May':        [0, 0, -25] + [0]*11,
    '-25 Jun':        [0, 0, 0, -25] + [0]*10,
    '-25x2 (Apr,Jun)':[0, -25, 0, -25] + [0]*10,
    '-25x2 (May,Jul)':[0, 0, -25, 0, -25] + [0]*9,
    '-25x3 (A,J,A)':  [0, -25, 0, -25, 0, -25] + [0]*8,
    '-25x3 (M,J,S)':  [0, 0, -25, 0, -25, 0, -25, 0] + [0]*6,
    '-50 Apr':        [0, -50] + [0]*12,
    '-25 Apr -50 Jun':[0, -25, 0, -50] + [0]*10,
    '+25 Apr':        [0, 25] + [0]*12,
    'Ease 25 each':   [0] + [-25]*13,
}

# --- WIDGETS ---
tibo_input = widgets.FloatText(value=4.75, description='TIBO (%)', step=0.05,
                                style={'description_width': '70px'},
                                layout=widgets.Layout(width='160px'))

mkt_inputs = {}
for label, _ in TENORS:
    mkt_inputs[label] = widgets.FloatText(
        value={'3M': 4.15, '6M': 4.10, '9M': 4.095, '12M': 4.095}[label],
        description=f'{label} Mkt%', step=0.005,
        style={'description_width': '70px'},
        layout=widgets.Layout(width='170px')
    )

# Path weight inputs and change-per-meeting inputs
path_names = list(DEFAULT_PATHS.keys())
weight_inputs = {}
path_change_inputs = {}  # dict of {path_name: [14 FloatText widgets]}

for pname in path_names:
    weight_inputs[pname] = widgets.FloatText(
        value=0.0, step=5.0, layout=widgets.Layout(width='60px'))
    path_change_inputs[pname] = []
    for j in range(14):
        w = widgets.FloatText(
            value=DEFAULT_PATHS[pname][j], step=25,
            layout=widgets.Layout(width='55px'))
        path_change_inputs[pname].append(w)

# Set some default weights that sum to 100
weight_inputs['Hold'].value = 10.0
weight_inputs['-25 Apr'].value = 15.0
weight_inputs['-25 Jun'].value = 15.0
weight_inputs['-25x2 (Apr,Jun)'].value = 25.0
weight_inputs['-25x2 (May,Jul)'].value = 10.0
weight_inputs['-25x3 (A,J,A)'].value = 15.0
weight_inputs['-25x3 (M,J,S)'].value = 10.0

# Tenor selector for chart
tenor_selector = widgets.Dropdown(
    options=[t[0] for t in TENORS],
    value='3M', description='Tenor:',
    style={'description_width': '50px'},
    layout=widgets.Layout(width='160px')
)

# Output areas
output_tables = widgets.Output()
output_chart = widgets.Output()

calc_button = widgets.Button(description='Calculate FV', button_style='info',
                              layout=widgets.Layout(width='140px', height='35px'))


# ============================================================
# CORE COMPUTATION
# ============================================================

def compute_all():
    tibo = tibo_input.value
    mkt = {label: mkt_inputs[label].value for label in mkt_inputs}
    
    # Gather paths and weights
    paths = {}
    weights = {}
    for pname in path_names:
        w = weight_inputs[pname].value
        chgs = [path_change_inputs[pname][j].value for j in range(14)]
        paths[pname] = chgs
        weights[pname] = w
    
    total_w = sum(weights.values())
    
    # Compute FV for each path x tenor
    fv_table = {}  # {path: {tenor: fv_rate}}
    for pname in path_names:
        fv_table[pname] = {}
        for label, _ in TENORS:
            mat = MATURITIES[label]
            fv = pen_ois_fair_value(tibo, MEETING_DATES, paths[pname], SETTLE, mat)
            fv_table[pname][label] = fv
    
    # Weighted FV
    weighted_fv = {}
    for label, _ in TENORS:
        if total_w > 0:
            weighted_fv[label] = sum(
                weights[p] / total_w * fv_table[p][label]
                for p in path_names if not np.isnan(fv_table[p][label])
            )
        else:
            weighted_fv[label] = np.nan
    
    # Find highest and lowest rate paths (by terminal rate)
    terminal_rates = {}
    for pname in path_names:
        terminal_rates[pname] = tibo + sum(paths[pname]) / 100.0
    
    active_paths = [p for p in path_names if weights[p] > 0]
    if active_paths:
        high_path = max(active_paths, key=lambda p: terminal_rates[p])
        low_path = min(active_paths, key=lambda p: terminal_rates[p])
    else:
        high_path = path_names[0]
        low_path = path_names[0]
    
    return {
        'tibo': tibo, 'mkt': mkt, 'paths': paths, 'weights': weights,
        'total_w': total_w, 'fv_table': fv_table, 'weighted_fv': weighted_fv,
        'high_path': high_path, 'low_path': low_path,
    }


def color_val(v, fmt='.3f', invert=False):
    if np.isnan(v): return '-'
    sign = -1 if invert else 1
    cls = 'pos' if v * sign >= 0 else 'neg'
    return f'<span class="{cls}">{v:{fmt}}</span>'


def build_tables_html(data):
    tenor_labels = [t[0] for t in TENORS]
    html = CSS
    html += '<div class="pen-ois-container">'
    html += '<div class="pen-ois-title">PEN OIS FAIR VALUE MODEL</div>'
    html += f'<div style="color:#aaa;font-size:11px;">Settle: {SETTLE} | TIBO: {data["tibo"]:.2f}% | Weights sum: {data["total_w"]:.0f}%</div>'
    
    # --- Meeting dates header ---
    html += '<br><div class="pen-ois-subtitle">MEETING DATES</div>'
    html += '<div style="font-size:11px;color:#aaa;">'
    for i, d in enumerate(MEETING_DATES):
        html += f'{i+1}. {d.strftime("%d-%b-%y")} &nbsp; '
        if (i+1) % 7 == 0: html += '<br>'
    html += '</div>'
    
    # --- FV TABLE ---
    html += '<br><div class="pen-ois-subtitle">FAIR VALUE TABLE</div>'
    html += '<table class="fv-table"><tr><th>Wt%</th><th>Scenario</th>'
    for t in tenor_labels:
        html += f'<th>{t}<br>{MATURITIES[t].strftime("%d-%b-%y")}</th>'
    html += '</tr>'
    
    for pname in path_names:
        w = data['weights'][pname]
        style = ' style="background:#1a2a3f;"' if w > 0 else ''
        html += f'<tr{style}><td>{w:.1f}</td><td style="text-align:left;">{pname}</td>'
        for t in tenor_labels:
            v = data['fv_table'][pname][t]
            html += f'<td>{v:.3f}</td>'
        html += '</tr>'
    
    # Mkt row
    html += '<tr style="background:#2a1a3f;"><td></td><td style="text-align:left;">Mkt</td>'
    for t in tenor_labels:
        html += f'<td>{data["mkt"][t]:.3f}</td>'
    html += '</tr>'
    
    # Weighted FV row
    html += '<tr class="fv-row"><td style="font-weight:bold;">'
    html += f'{data["total_w"]:.0f}</td><td style="text-align:left;font-weight:bold;color:{GOLD};">FV</td>'
    for t in tenor_labels:
        html += f'<td style="color:{GOLD};">{data["weighted_fv"][t]:.3f}</td>'
    html += '</tr>'
    
    # Mkt - FV (bps)
    html += '<tr class="delta-row"><td></td><td style="text-align:left;">Mkt - FV (bp)</td>'
    for t in tenor_labels:
        delta_bp = (data['mkt'][t] - data['weighted_fv'][t]) * 100
        html += f'<td>{color_val(delta_bp, ".1f")}</td>'
    html += '</tr>'
    
    # High delta active
    hp = data['high_path']
    html += f'<tr class="delta-row" style="background:#004d40;"><td></td>'
    html += f'<td style="text-align:left;">High Δ Active<br><small>({hp})</small></td>'
    for t in tenor_labels:
        d = (data['mkt'][t] - data['fv_table'][hp][t]) * 100
        html += f'<td>{color_val(d, ".1f")}</td>'
    html += '</tr>'
    
    # Low delta active
    lp = data['low_path']
    html += f'<tr class="delta-row" style="background:#1a237e;"><td></td>'
    html += f'<td style="text-align:left;">Low Δ Active<br><small>({lp})</small></td>'
    for t in tenor_labels:
        d = (data['mkt'][t] - data['fv_table'][lp][t]) * 100
        html += f'<td>{color_val(d, ".1f")}</td>'
    html += '</tr>'
    
    html += '</table>'
    
    # --- PROBABILITY TABLE ---
    html += '<br><div class="pen-ois-subtitle">IMPLIED PROBABILITY OF 25bp MOVES (from Mkt vs Paths)</div>'
    html += '<table class="fv-table"><tr><th>Metric</th>'
    for i, d in enumerate(MEETING_DATES[:8]):
        html += f'<th>{d.strftime("%d-%b-%y")}</th>'
    html += '</tr>'
    
    # Use the 3M OIS as primary gauge for near-term probabilities
    # Simplified probability: compare market rate to hold vs cut scenarios
    tibo = data['tibo']
    mkt_3m = data['mkt']['3M']
    hold_fv = data['fv_table']['Hold']['3M']
    
    # Cumulative cuts implied
    # rate_diff = hold_fv - mkt_3m (how much lower mkt is vs hold = cuts priced)
    cum_cuts_bp = (hold_fv - mkt_3m) * 100
    
    # For each meeting, compute the forward rate implied cut probability
    # Build the step schedule: rate at each meeting boundary
    meeting_cut_probs = []
    cum_cuts = []
    running_cum = 0
    
    for i in range(min(8, len(MEETING_DATES))):
        # Create a path with -25 only at meeting i
        path_cut_i = [0]*14
        path_cut_i[i] = -25
        # FV if only this meeting cuts
        fv_cut = pen_ois_fair_value(tibo, MEETING_DATES, path_cut_i, SETTLE, MATURITIES['3M'])
        fv_hold = pen_ois_fair_value(tibo, MEETING_DATES, [0]*14, SETTLE, MATURITIES['3M'])
        
        # Sensitivity of 3M OIS to a cut at this meeting
        sens = fv_hold - fv_cut  # positive if cut lowers rate
        if sens > 0.001:
            # Implied probability
            p = min(max((hold_fv - mkt_3m) / (sens) * 100, -100), 200)
        else:
            p = 0
        
        # Use a simpler cumulative approach across tenors
        # Cumulative cuts at each meeting implied by longer tenors
        cum_path = [0]*14
        for j in range(i+1):
            cum_path[j] = -25
        
        # Check how many tenors this meeting falls within
        best_tenor = '3M'
        for label, mo in TENORS:
            if MEETING_DATES[i] < MATURITIES[label]:
                best_tenor = label
                break
        
        fv_cum_cut = pen_ois_fair_value(tibo, MEETING_DATES, cum_path, SETTLE, MATURITIES[best_tenor])
        fv_no_cut = pen_ois_fair_value(tibo, MEETING_DATES, [0]*14, SETTLE, MATURITIES[best_tenor])
        mkt_t = data['mkt'][best_tenor]
        
        spread = fv_no_cut - fv_cum_cut
        if spread > 0.001:
            cum_p = (fv_no_cut - mkt_t) / spread * 100
        else:
            cum_p = 0
        
        meeting_cut_probs.append(cum_p / max(i+1, 1))  # marginal
        cum_cuts.append(cum_p)
    
    # Marginal prob row
    html += '<tr><td style="text-align:left;">P(cut 25bp)</td>'
    for i in range(min(8, len(meeting_cut_probs))):
        p = meeting_cut_probs[i]
        html += f'<td>{color_val(p, ".0f")}%</td>'
    html += '</tr>'
    
    # Cumulative cuts row
    html += '<tr><td style="text-align:left;">Cum cuts (%)</td>'
    for i in range(min(8, len(cum_cuts))):
        html += f'<td>{color_val(cum_cuts[i], ".0f")}%</td>'
    html += '</tr>'
    
    # Cumulative bp row
    html += '<tr><td style="text-align:left;">Cum cuts (bp)</td>'
    for i in range(min(8, len(cum_cuts))):
        bp = cum_cuts[i] / 100 * 25 * (i+1)
        html += f'<td>{color_val(bp, ".1f")}</td>'
    html += '</tr>'
    
    html += '</table>'
    html += '</div>'
    return html


def build_chart(data, selected_tenor):
    """Build the Plotly chart for a selected tenor."""
    label = selected_tenor
    mat = MATURITIES[label]
    mkt_rate = data['mkt'][label]
    fv_rate = data['weighted_fv'][label]
    
    # Generate simulated history
    hist_dates, hist_rates = generate_simulated_history(mkt_rate, n_days=252)
    
    fig = make_subplots(rows=1, cols=1)
    
    # Historical candlestick-like line
    fig.add_trace(go.Scatter(
        x=hist_dates, y=hist_rates,
        mode='lines', name='Historical',
        line=dict(color='#4fc3f7', width=1.5),
    ))
    
    # FV line
    fig.add_hline(y=fv_rate, line_dash='dash', line_color=GOLD, line_width=2,
                  annotation_text=f'FV: {fv_rate:.3f}',
                  annotation_position='right',
                  annotation_font_color=GOLD)
    
    # Market line
    fig.add_hline(y=mkt_rate, line_dash='solid', line_color=RED, line_width=2,
                  annotation_text=f'Mkt: {mkt_rate:.3f}',
                  annotation_position='right',
                  annotation_font_color=RED)
    
    # Path scenario FV lines for active paths
    colors = ['#e040fb', '#69f0ae', '#ffd54f', '#4fc3f7', '#ff8a65',
              '#ce93d8', '#80cbc4', '#fff176', '#ef9a9a', '#a5d6a7', '#90caf9', '#ffcc80']
    ci = 0
    for pname in path_names:
        if data['weights'][pname] > 0:
            pv = data['fv_table'][pname][label]
            fig.add_hline(
                y=pv, line_dash='dot', line_width=1,
                line_color=colors[ci % len(colors)],
                annotation_text=f'{pname}: {pv:.3f}',
                annotation_position='left',
                annotation_font_color=colors[ci % len(colors)],
                annotation_font_size=10
            )
            ci += 1
    
    # High/Low path bands
    hp_fv = data['fv_table'][data['high_path']][label]
    lp_fv = data['fv_table'][data['low_path']][label]
    
    fig.update_layout(
        title=dict(text=f'PEN OIS {label} ({mat}) — FV Levels Overlay', font=dict(color=ACCENT, size=16)),
        template='plotly_dark',
        paper_bgcolor=DARK_BG,
        plot_bgcolor='#0d1f3c',
        height=500,
        margin=dict(l=50, r=180, t=60, b=40),
        yaxis=dict(title='Rate (%)', gridcolor='#1a3a5c'),
        xaxis=dict(gridcolor='#1a3a5c'),
        legend=dict(font=dict(size=10)),
        font=dict(family='Consolas, monospace'),
    )
    
    return fig


def build_rate_path_chart(data):
    """Build chart showing TIBO rate evolution under each active scenario."""
    tibo = data['tibo']
    fig = go.Figure()
    
    colors = ['#e040fb', '#69f0ae', '#ffd54f', '#4fc3f7', '#ff8a65',
              '#ce93d8', '#80cbc4', '#fff176', '#ef9a9a', '#a5d6a7', '#90caf9', '#ffcc80']
    ci = 0
    
    for pname in path_names:
        if data['weights'][pname] > 0:
            # Build TIBO path
            dates_p = [SETTLE]
            rates_p = [tibo]
            r = tibo
            for i, mtg in enumerate(MEETING_DATES):
                eff = mtg + timedelta(days=1)
                dates_p.append(eff)
                rates_p.append(r)  # rate before change
                r += data['paths'][pname][i] / 100.0
                dates_p.append(eff)
                rates_p.append(r)  # rate after change
            
            fig.add_trace(go.Scatter(
                x=dates_p, y=rates_p,
                mode='lines', name=f'{pname} ({data["weights"][pname]:.0f}%)',
                line=dict(color=colors[ci % len(colors)], width=2),
            ))
            ci += 1
    
    # Add meeting date markers
    fig.add_trace(go.Scatter(
        x=[d for d in MEETING_DATES],
        y=[tibo]*len(MEETING_DATES),
        mode='markers', name='Meetings',
        marker=dict(symbol='diamond', size=8, color=ACCENT),
        text=[d.strftime('%d-%b-%y') for d in MEETING_DATES],
    ))
    
    fig.update_layout(
        title=dict(text='BCRP Reference Rate (TIBO) — Scenario Paths', font=dict(color=ACCENT, size=16)),
        template='plotly_dark',
        paper_bgcolor=DARK_BG,
        plot_bgcolor='#0d1f3c',
        height=400,
        margin=dict(l=50, r=50, t=60, b=40),
        yaxis=dict(title='TIBO Rate (%)', gridcolor='#1a3a5c'),
        xaxis=dict(gridcolor='#1a3a5c'),
        legend=dict(font=dict(size=10)),
        font=dict(family='Consolas, monospace'),
    )
    return fig


# ============================================================
# EVENT HANDLERS
# ============================================================

def on_calculate(b=None):
    data = compute_all()
    
    with output_tables:
        clear_output(wait=True)
        display(HTML(build_tables_html(data)))
    
    with output_chart:
        clear_output(wait=True)
        fig1 = build_chart(data, tenor_selector.value)
        fig1.show()
        fig2 = build_rate_path_chart(data)
        fig2.show()

calc_button.on_click(on_calculate)
tenor_selector.observe(lambda change: on_calculate() if change['name'] == 'value' else None)


# ============================================================
# LAYOUT
# ============================================================

# Market inputs row
mkt_box = widgets.HBox(
    [tibo_input] + [mkt_inputs[t] for t, _ in TENORS],
    layout=widgets.Layout(margin='5px 0')
)

# Path input grid
meeting_headers = [widgets.Label(MEETING_DATES[j].strftime('%b-%y'),
                   layout=widgets.Layout(width='55px', min_width='55px'))
                   for j in range(14)]

path_rows = []
# Header row
header_row = widgets.HBox(
    [widgets.Label('Wt%', layout=widgets.Layout(width='60px')),
     widgets.Label('Scenario', layout=widgets.Layout(width='130px'))] +
    meeting_headers,
    layout=widgets.Layout(margin='2px 0')
)
path_rows.append(header_row)

for pname in path_names:
    row = widgets.HBox(
        [weight_inputs[pname],
         widgets.Label(pname, layout=widgets.Layout(width='130px', min_width='130px'))] +
        path_change_inputs[pname],
        layout=widgets.Layout(margin='1px 0')
    )
    path_rows.append(row)

paths_box = widgets.VBox(path_rows, layout=widgets.Layout(
    border='1px solid #1a3a5c', padding='5px', overflow_x='auto'
))

controls = widgets.HBox([calc_button, tenor_selector],
                         layout=widgets.Layout(margin='8px 0'))

# Full layout
display(HTML(f'<div style="background:{DARK_BG};padding:10px;">'
             f'<span style="color:{ACCENT};font-size:22px;font-weight:bold;">'
             f'PEN OIS FAIR VALUE MODEL</span>'
             f'<br><span style="color:#aaa;font-size:12px;">'
             f'ACT/360 | Zero Coupon | Annual | Nominal Rate Convention</span></div>'))

display(HTML(f'<div style="color:{GOLD};font-weight:bold;margin:6px 0;">MARKET INPUTS</div>'))
display(mkt_box)

display(HTML(f'<div style="color:{GOLD};font-weight:bold;margin:6px 0;">'
             f'INPUT PATHS — Rate change (bps) per BCRP meeting (max 14)</div>'))
display(paths_box)

display(controls)
display(output_tables)
display(output_chart)

# Auto-calculate on load
on_calculate()